# Hashmaps
In this notebook, we will explore the concept of hashmaps, which are a fundamental data structure in computer science. Hashmaps allow for efficient data retrieval and storage by using a hash function to map keys to values.

In python, we have a built-in dictionary data structure that functions as a hashmap. But we will implement our own hashmap to understand how it works under the hood.

# Let's start by defining a simple HashMap class

In [1]:
from collections.abc import MutableMapping

# abc: Abstract Base Class

class Mapbase(MutableMapping):
    def __init__(self, k, v):
        self._key = k
        self._value = v
    def __eq__(self, other):
        return self._value == other._value 
    def __ne__(self, other):
        return not (self == other)
    def __lt__(self, other):
        return self._value < other._value

class UnsortedTableMap(Mapbase):
    class _Item:
        def __init__(self, k, v):
            self._key = k
            self._value = v

    def __init__(self):
        self._table = []
    def __len__(self):
        return len(self._table)
    def __getitem__(self, k):
        for item in self._table:
            if item._key == k:
                return item._value
        raise KeyError(f'Key Error: {k}')
    def __setitem__(self, k, v):
        for item in self._table:
            if item._key == k:
                item._value = v
                return
        self._table.append(self._Item(k, v))    
    def __delitem__(self, k):
        for j in range(len(self._table)):
            if k == self._table[j]._key:
                self._table.pop(j)
                return
        raise KeyError(f'Key Error: {k}')
    def __iter__(self):
        for item in self._table:
            yield item._key

In [2]:
data = UnsortedTableMap()
len(data)
data[5] = 'five'
data[3] = 'three'
data[7] = 'seven'
data[5] = 'five updated'
data[3] = 'three updated'
data[7] = 'seven updated'
for _value in data:
    print(_value, data[_value])
data[5]
data[3]
data[7]
data[5] = 'five updated again'
len(data)

5 five updated
3 three updated
7 seven updated


3

In [ ]:
hash("hello")

6536840939547686264

In [7]:
from random import randrange

class HashmapBase(Mapbase):
    def __init__(self, cap=11, p=109345121):
        self._table = [None] * cap
        self._n = 0
        self._prime = p
        self._scale = 1 + randrange(p-1)
        self._shift = randrange(p)
    def _hash_function(self, k):
        return (hash(k) * self._scale + self._shift) % self._prime % len(self._table)
    def __len__(self):
        return self._n
    def __getitem__(self, k):
        j = self._hash_function(k)
        return self._bucket_getitem(j, k)
    def __setitem__(self, k, v):
        j = self._hash_function(k)
        self._bucket_setitem(j, k, v)
        if self._n > len(self._table) // 2:
            self._resize(2 * len(self._table) - 1)
    def __delitem__(self, k):
        j = self._hash_function(k)
        self._bucket_delitem(j, k)
        self._n -= 1
    def _resize(self, c):
        old = list(self.items())
        self._table = [None] * c
        self._n = 0
        for (k, v) in old:
            self[k] = v